# HoTHP Grid Search: Decaimento LENTO (controle)

**Como rodar:** Runtime -> Change runtime type -> T4 GPU (ou L4)

---

## Objetivo

Confirmar que o HoTHP **nao** tem vantagem no cenario de decaimento lento.
Isso eh o experimento controle do grid search de decaimento rapido.

Se os resultados mostrarem vantagem ~0 e nao significativa, confirmamos que
a vantagem do HoTHP eh especifica para processos de decaimento rapido.

**Tempo estimado:** ~2-3h numa T4

In [ ]:
# ============================================================================
# CELULA 1 — Instalacao
# ============================================================================

import os

if not os.path.exists('ufc-easytpp'):
    !git clone https://github.com/hugoramos/ufc-easytpp.git

!pip install omegaconf -q

# Fixes
_init_path = 'ufc-easytpp/easy_tpp/model/__init__.py'
with open(_init_path, 'w') as f:
    f.write("""from easy_tpp.model.torch_model.torch_basemodel import TorchBaseModel
from easy_tpp.model.torch_model.torch_thp import THP as TorchTHP
from easy_tpp.model.torch_model.torch_rothp import RoTHP as TorchRoTHP
from easy_tpp.model.torch_model.torch_hothp import HoTHP as TorchHoTHP
""")

_hothp_path = 'ufc-easytpp/easy_tpp/model/torch_model/torch_hothp.py'
with open(_hothp_path, 'r') as f:
    code = f.read()
if 'from notebooks.' in code:
    code = code.replace(
        'from notebooks.Extrapolation_and_Attention_Analysis import attention_fixed\n', '')
    with open(_hothp_path, 'w') as f:
        f.write(code)

print('OK')

In [ ]:
# ============================================================================
# CELULA 2 — Imports e configuracao (DECAIMENTO LENTO)
# ============================================================================

import os, sys, math, random, hashlib, contextlib, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from torch.utils.data import DataLoader
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# ── Grid de busca ─────────────────────────────────────────────────────────

TRAIN_LENS     = [10, 20, 50, 100]
EXTRAP_FACTORS = [2, 5, 10]
N_SEEDS        = 5
EPOCHS         = 500
PATIENCE       = 30
BASE_SEED      = 42
NUM_TYPES      = 2
PAD_ID         = NUM_TYPES

# DECAIMENTO LENTO (controle)
PROC = dict(
    mu    = np.array([0.3, 0.3]),
    alpha = np.array([[0.008, 0.006], [0.006, 0.008]]),
    beta  = 0.02,
)

COR_ROTHP = '#4C72B0'
COR_HOTHP = '#C44E52'

# ── Reproducibilidade ─────────────────────────────────────────────────────

def set_seed(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)

def run_seed(*parts):
    key = '::'.join(map(str, parts))
    return (BASE_SEED + int(hashlib.sha256(key.encode()).hexdigest()[:8], 16)) % (2**31)

set_seed(BASE_SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = device.type == 'cuda'

if USE_AMP:
    try:
        _autocast = lambda: torch.amp.autocast(device_type='cuda')
        _Scaler   = torch.amp.GradScaler
    except AttributeError:
        _autocast = torch.cuda.amp.autocast
        _Scaler   = torch.cuda.amp.GradScaler
else:
    _autocast = contextlib.nullcontext
    _Scaler   = None

sys.path.insert(0, os.path.abspath('ufc-easytpp'))

import easy_tpp.model.torch_model.torch_baselayer as baselayer

def _attention(query, key, value, mask=None, dropout=None):
    d_k = query.size(-1)
    scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        if mask.dim() == 3:
            mask = mask.unsqueeze(1)
        scores = scores.masked_fill(mask > 0, -1e4)
    p = torch.softmax(scores, dim=-1)
    if dropout is not None:
        p = dropout(p)
    return torch.matmul(p, value), p

baselayer.attention = _attention
import easy_tpp.model.torch_model.torch_rothp as rothp_module
rothp_module.attention = _attention

from easy_tpp.config_factory.model_config import ModelConfig
from easy_tpp.model.torch_model.torch_rothp import RoTHP
from easy_tpp.model.torch_model.torch_hothp import HoTHP

max_test = TRAIN_LENS[-1] * EXTRAP_FACTORS[-1]

print(f'Device: {device}  |  AMP: {USE_AMP}')
print(f'PROCESSO: DECAIMENTO LENTO (beta={PROC["beta"]})')
print(f'TRAIN_LENS: {TRAIN_LENS}')
print(f'EXTRAP_FACTORS: {EXTRAP_FACTORS}')
print(f'Maior sequencia de teste: {max_test} eventos')
print(f'Seeds: {N_SEEDS}')

In [ ]:
# ============================================================================
# CELULA 3 — Funcoes auxiliares (simulacao, collate, treino)
# ============================================================================

def simulate_hawkes(rng, mu, alpha, beta, horizon, min_ev, max_ev):
    for _ in range(100):
        events, t = [], 0.0
        while t < horizon and len(events) < max_ev:
            lam = mu.copy()
            for ti, ki in events:
                lam += alpha[:, ki] * np.exp(-beta * (t - ti))
            lam_bar = float(lam.sum())
            if lam_bar < 1e-9:
                break
            t += rng.exponential(1.0 / lam_bar)
            if t >= horizon:
                break
            cand = mu.copy()
            for ti, ki in events:
                cand += alpha[:, ki] * np.exp(-beta * (t - ti))
            if rng.uniform() <= cand.sum() / lam_bar:
                probs = cand / cand.sum()
                events.append((t, int(rng.choice(len(mu), p=probs))))
        if len(events) >= min_ev:
            return events[:max_ev]
    return events[:max_ev]


def to_tensors(seqs):
    out = []
    for seq in seqs:
        seq = sorted(seq, key=lambda x: x[0])
        t = torch.tensor([x[0] for x in seq], dtype=torch.float32)
        k = torch.tensor([x[1] for x in seq], dtype=torch.long)
        d = torch.zeros_like(t)
        d[1:] = t[1:] - t[:-1]
        mg = d[1:].mean().clamp(min=1e-6)
        t = (t - t[0]) / mg
        d = d / mg
        out.append({'time_seqs': t, 'time_delta_seqs': d, 'type_seqs': k})
    return out


def collate(batch, pad_id=PAD_ID):
    B = len(batch)
    L = max(len(x['time_seqs']) for x in batch)
    t_pad = torch.zeros(B, L)
    d_pad = torch.zeros(B, L)
    k_pad = torch.full((B, L), pad_id, dtype=torch.long)
    npm   = torch.zeros(B, L)
    causal = torch.triu(torch.ones(L, L, dtype=torch.bool), diagonal=1)
    attn   = torch.ones(B, L, L, dtype=torch.bool)
    for i, item in enumerate(batch):
        sl = len(item['time_seqs'])
        t_pad[i, :sl] = item['time_seqs']
        d_pad[i, :sl] = item['time_delta_seqs']
        k_pad[i, :sl] = item['type_seqs']
        npm[i, :sl] = 1.0
        m = causal.clone()
        m[:, sl:] = True
        m[sl:, :] = True
        attn[i] = m
    return t_pad, d_pad, k_pad, npm, attn


def make_loader(data, bs, shuffle=False, seed=None):
    g = None
    if shuffle and seed is not None:
        g = torch.Generator()
        g.manual_seed(seed)
    return DataLoader(data, batch_size=bs, shuffle=shuffle,
                      collate_fn=collate, generator=g)


config = ModelConfig(**{
    'hidden_size': 32, 'num_layers': 2, 'num_heads': 2, 'dropout_rate': 0.1,
    'num_event_types': NUM_TYPES, 'num_event_types_pad': NUM_TYPES + 1,
    'event_pad_index': PAD_ID, 'time_emb_size': 32, 'use_ln': True,
    'gpu': 0 if torch.cuda.is_available() else -1,
    'model_id': 'Grid',
    'thinning': {'num_sample': 1, 'num_exp': 500, 'over_sample_rate': 5.0,
                 'patience_counter': 5, 'num_samples_boundary': 5,
                 'dtime_max': 5.0, 'num_step_gen': 1},
    'loss_integral_num_sample_per_step': 20,
    'use_mc_samples': False,
})


def eval_nll(model, dl):
    model.eval()
    total_l = total_n = 0
    with torch.no_grad():
        for batch in dl:
            batch = [t.to(device) for t in batch]
            with _autocast():
                l, n = model.loglike_loss(batch)
            total_l += l.item()
            total_n += n
    return total_l / (total_n + 1e-9)


def train_model(cls, train_dl, val_dl, lr, base_seed):
    set_seed(base_seed)
    m = cls(config).to(device)
    opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode='min', factor=0.5, patience=10, min_lr=1e-5)
    scaler = _Scaler(enabled=True) if USE_AMP else None
    best_val = float('inf')
    best_state = None
    no_imp = 0
    for ep in range(EPOCHS):
        m.train()
        for batch in train_dl:
            batch = [t.to(device) for t in batch]
            opt.zero_grad()
            with _autocast():
                l, n = m.loglike_loss(batch)
                nll = l / (n + 1e-9)
            if not torch.isnan(nll):
                if scaler:
                    scaler.scale(nll).backward()
                    scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
                    scaler.step(opt)
                    scaler.update()
                else:
                    nll.backward()
                    torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
                    opt.step()
        v = eval_nll(m, val_dl)
        sched.step(v)
        if v < best_val - 1e-4:
            best_val = v
            best_state = {k: v2.cpu().clone() for k, v2 in m.state_dict().items()}
            no_imp = 0
        else:
            no_imp += 1
        if no_imp >= PATIENCE:
            break
    m.load_state_dict(best_state)
    return m, best_val


print('Funcoes prontas.')

In [ ]:
# ============================================================================
# CELULA 4 — Gerar dados (com cache em disco) - DECAIMENTO LENTO
# ============================================================================

import pickle

DATA_CACHE = 'grid_data_cache_slow.pkl'

mu, alpha, beta = PROC['mu'], PROC['alpha'], PROC['beta']

if os.path.exists(DATA_CACHE):
    print(f'Cache encontrado: {DATA_CACHE}. Carregando...')
    with open(DATA_CACHE, 'rb') as f:
        all_datasets = pickle.load(f)
    print(f'Carregado. TRAIN_LENS disponiveis: {list(all_datasets.keys())}')
else:
    print('Nenhum cache. Gerando dados do zero (decaimento lento)...')
    rng = np.random.default_rng(run_seed('grid_slow', 'data'))
    all_datasets = {}

    for tl in TRAIN_LENS:
        print(f'Gerando dados para TRAIN_LEN={tl}...')
        raw_train = [simulate_hawkes(rng, mu, alpha, beta, 50.0 * (tl/50), 5, tl)
                     for _ in range(500)]
        raw_val   = [simulate_hawkes(rng, mu, alpha, beta, 50.0 * (tl/50), 5, tl)
                     for _ in range(150)]
        raw_short = [simulate_hawkes(rng, mu, alpha, beta, 50.0 * (tl/50), 5, tl)
                     for _ in range(200)]

        ds = {
            'train': to_tensors(raw_train),
            'val':   to_tensors(raw_val),
            'short': to_tensors(raw_short),
        }

        for f in EXTRAP_FACTORS:
            target_len = tl * f
            raw_ext = [simulate_hawkes(rng, mu, alpha, beta,
                                       50.0 * (target_len/50),
                                       tl + 5, target_len)
                       for _ in range(200)]
            ds[f'extrap_{f}x'] = to_tensors(raw_ext)
            mean_len = np.mean([len(s['time_seqs']) for s in ds[f'extrap_{f}x']])
            print(f'  {f}x: target={target_len}, mean_len={mean_len:.0f}')

        all_datasets[tl] = ds

    with open(DATA_CACHE, 'wb') as f:
        pickle.dump(all_datasets, f)
    print(f'\nDados salvos em {DATA_CACHE}')

print('Dados prontos.')

In [ ]:
# ============================================================================
# CELULA 5 — Execucao do grid search - DECAIMENTO LENTO
# ============================================================================

import gc

CHECKPOINT_FILE = 'grid_checkpoint_slow.csv'

seeds = [BASE_SEED + i * 100 for i in range(N_SEEDS)]

def get_eval_bs(test_len):
    if test_len <= 200:
        return 16
    elif test_len <= 500:
        return 8
    elif test_len <= 1000:
        return 4
    else:
        return 2

if os.path.exists(CHECKPOINT_FILE):
    df_done = pd.read_csv(CHECKPOINT_FILE)
    all_results = df_done.to_dict('records')
    done_keys = set(zip(df_done['train_len'], df_done['factor'], df_done['seed']))
    print(f'Checkpoint encontrado: {len(all_results)} resultados ja computados.')
else:
    all_results = []
    done_keys = set()
    print('Nenhum checkpoint encontrado. Comecando do zero.')

total = len(TRAIN_LENS) * N_SEEDS
count = 0

for tl in TRAIN_LENS:
    ds = all_datasets[tl]

    tl_remaining = [(seed, f) for seed in seeds for f in EXTRAP_FACTORS
                    if (tl, f, seed) not in done_keys]
    if len(tl_remaining) == 0:
        print(f'TRAIN_LEN={tl}: ja computado, pulando.')
        count += N_SEEDS
        continue

    val_dl   = make_loader(ds['val'],   64)
    short_dl = make_loader(ds['short'], 64)
    ext_dls = {f: make_loader(ds[f'extrap_{f}x'], get_eval_bs(tl * f))
               for f in EXTRAP_FACTORS}

    for seed_idx, seed in enumerate(seeds):
        seed_remaining = [f for f in EXTRAP_FACTORS if (tl, f, seed) not in done_keys]
        if len(seed_remaining) == 0:
            count += 1
            continue

        count += 1
        print(f'[{count}/{total}] TRAIN_LEN={tl}, Seed {seed_idx+1}/{N_SEEDS}', end='  ')

        train_dl = make_loader(ds['train'], 64, shuffle=True,
                               seed=run_seed(tl, seed))

        rothp, rv = train_model(RoTHP, train_dl, val_dl, lr=1e-3,
                                base_seed=run_seed(tl, 'rothp', seed))
        hothp, hv = train_model(HoTHP, train_dl, val_dl, lr=5e-4,
                                base_seed=run_seed(tl, 'hothp', seed))

        r_short = eval_nll(rothp, short_dl)
        h_short = eval_nll(hothp, short_dl)

        print(f'val: R={rv:.4f} H={hv:.4f}')

        for f in EXTRAP_FACTORS:
            if (tl, f, seed) in done_keys:
                continue

            test_len = tl * f
            try:
                r_full = eval_nll(rothp, ext_dls[f])
                h_full = eval_nll(hothp, ext_dls[f])
            except RuntimeError as e:
                if 'out of memory' in str(e).lower():
                    print(f'    OOM em TRAIN_LEN={tl} x{f}, pulando.')
                    torch.cuda.empty_cache()
                    gc.collect()
                    r_full = float('nan')
                    h_full = float('nan')
                else:
                    raise

            all_results.append({
                'train_len': tl, 'factor': f, 'test_len': test_len,
                'seed': seed, 'r_short': r_short, 'h_short': h_short,
                'r_full': r_full, 'h_full': h_full,
                'adv': r_full - h_full, 'r_val': rv, 'h_val': hv,
            })

        del rothp, hothp
        torch.cuda.empty_cache()
        gc.collect()

    df = pd.DataFrame(all_results)
    df.to_csv(CHECKPOINT_FILE, index=False)
    print(f'  -> Checkpoint salvo: {len(all_results)} resultados')

df = pd.DataFrame(all_results)
print(f'\nGrid search concluido! {len(df)} resultados totais.')

In [ ]:
# ============================================================================
# CELULA 6 — Tabela resumo (DECAIMENTO LENTO)
# ============================================================================

print('=' * 85)
print('GRID SEARCH: Vantagem do HoTHP (NLL_RoTHP - NLL_HoTHP)')
print('Positivo = HoTHP melhor  |  Processo: DECAIMENTO LENTO  |  n=5 seeds')
print('=' * 85)
print()

for tl in TRAIN_LENS:
    sub = df[df['train_len'] == tl]
    print(f'TRAIN_LEN = {tl}')
    print(f'  {"Factor":>6s}  {"Test len":>8s}  {"NLL RoTHP":>14s}  {"NLL HoTHP":>14s}  {"Vantagem":>14s}  {"p":>7s}  Sig')
    print(f'  {"-"*6}  {"-"*8}  {"-"*14}  {"-"*14}  {"-"*14}  {"-"*7}  ---')
    
    for f in EXTRAP_FACTORS:
        rows = sub[sub['factor'] == f]
        rm = rows['r_full'].mean()
        rs = rows['r_full'].std()
        hm = rows['h_full'].mean()
        hs = rows['h_full'].std()
        adv = rows['adv'].values
        
        t_stat, p2 = stats.ttest_1samp(adv, 0)
        p1 = p2 / 2 if t_stat > 0 else 1.0
        sig = '***' if p1 < 0.001 else '**' if p1 < 0.01 else '*' if p1 < 0.05 else '~' if p1 < 0.10 else 'ns'
        
        print(f'  {f:>5}x  {tl*f:>8d}  {rm:.4f}+/-{rs:.4f}  {hm:.4f}+/-{hs:.4f}  {adv.mean():>+.4f}+/-{adv.std():.4f}  p={p1:.3f}  {sig}')
    print()

In [ ]:
# ============================================================================
# CELULA 7 — Heatmap: vantagem do HoTHP (TRAIN_LEN x Fator)
# ============================================================================

# Monta matriz de vantagem media
adv_matrix = np.zeros((len(TRAIN_LENS), len(EXTRAP_FACTORS)))
sig_matrix = np.empty((len(TRAIN_LENS), len(EXTRAP_FACTORS)), dtype=object)

for i, tl in enumerate(TRAIN_LENS):
    for j, f in enumerate(EXTRAP_FACTORS):
        rows = df[(df['train_len'] == tl) & (df['factor'] == f)]
        adv = rows['adv'].values
        adv_matrix[i, j] = adv.mean()
        
        t_stat, p2 = stats.ttest_1samp(adv, 0)
        p1 = p2 / 2 if t_stat > 0 else 1.0
        if p1 < 0.001:
            sig_matrix[i, j] = '***'
        elif p1 < 0.01:
            sig_matrix[i, j] = '**'
        elif p1 < 0.05:
            sig_matrix[i, j] = '*'
        else:
            sig_matrix[i, j] = ''

# Anotacoes combinando valor + significancia
annot = np.empty_like(adv_matrix, dtype=object)
for i in range(adv_matrix.shape[0]):
    for j in range(adv_matrix.shape[1]):
        annot[i, j] = f'{adv_matrix[i,j]:+.3f}\n{sig_matrix[i,j]}'

fig, ax = plt.subplots(figsize=(10, 6))

sns.heatmap(adv_matrix, annot=annot, fmt='',
            xticklabels=[f'{f}x ({tl*f} ev)' for f in EXTRAP_FACTORS for tl in [TRAIN_LENS[0]]],
            yticklabels=[f'L={tl}' for tl in TRAIN_LENS],
            cmap='RdYlGn', center=0, linewidths=1, linecolor='white',
            cbar_kws={'label': 'Vantagem HoTHP (nats)'},
            ax=ax)

# Corrige xticklabels para mostrar test_len correto por linha
ax.set_xticklabels([f'{f}x' for f in EXTRAP_FACTORS], fontsize=12)
ax.set_yticklabels([f'L={tl}' for tl in TRAIN_LENS], fontsize=12, rotation=0)

ax.set_xlabel('Fator de extrapolacao', fontsize=13)
ax.set_ylabel('TRAIN_LEN (eventos no treino)', fontsize=13)
ax.set_title('Vantagem do HoTHP sobre RoTHP (NLL)\n'
             'Verde = HoTHP melhor  |  Vermelho = RoTHP melhor\n'
             f'Processo: decaimento rapido  |  n={N_SEEDS} seeds',
             fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('grid_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================================
# CELULA 8 — Grafico de linhas: NLL por fator para cada TRAIN_LEN
# ============================================================================

fig, axes = plt.subplots(1, len(TRAIN_LENS), figsize=(5 * len(TRAIN_LENS), 5))

x = np.arange(len(EXTRAP_FACTORS))

for ax, tl in zip(axes, TRAIN_LENS):
    sub = df[df['train_len'] == tl]
    
    for col, label, color in [
        ('r_full', 'RoTHP', COR_ROTHP),
        ('h_full', 'HoTHP', COR_HOTHP),
    ]:
        means = []
        stds = []
        for f in EXTRAP_FACTORS:
            vals = sub[sub['factor'] == f][col].values
            means.append(vals.mean())
            stds.append(vals.std())
        means = np.array(means)
        stds = np.array(stds)
        
        ax.plot(x, means, 'o-', color=color, lw=2, ms=7, label=label)
        ax.fill_between(x, means - stds, means + stds, color=color, alpha=0.15)
    
    # NLL in-dist como referencia
    r_short_mean = sub.drop_duplicates('seed')['r_short'].mean()
    h_short_mean = sub.drop_duplicates('seed')['h_short'].mean()
    ax.axhline(r_short_mean, color=COR_ROTHP, ls=':', lw=1, alpha=0.5)
    ax.axhline(h_short_mean, color=COR_HOTHP, ls=':', lw=1, alpha=0.5)
    
    ax.set_xticks(x)
    ax.set_xticklabels([f'{f}x\n({tl*f} ev)' for f in EXTRAP_FACTORS], fontsize=10)
    ax.set_xlabel('Fator de extrapolacao')
    ax.set_ylabel('NLL (nats)')
    ax.set_title(f'TRAIN_LEN = {tl}', fontweight='bold', fontsize=13)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

fig.suptitle('NLL por Fator de Extrapolacao\n'
             'Linhas pontilhadas = NLL in-dist  |  Processo: decaimento rapido',
             fontsize=13, fontweight='bold', y=1.04)
plt.tight_layout()
plt.savefig('grid_nll_lines.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================================
# CELULA 9 — Heatmap do test_len absoluto (TRAIN_LEN x test_len)
# ============================================================================

# Coleta todos os test_lens unicos
all_test_lens = sorted(df['test_len'].unique())

print('Mapa completo: TRAIN_LEN vs TEST_LEN absoluto')
print(f'  {"":>10s}', end='')
for test_l in all_test_lens:
    print(f'  {test_l:>6d}', end='')
print()

for tl in TRAIN_LENS:
    print(f'  L={tl:>4d}  ', end='')
    for test_l in all_test_lens:
        rows = df[(df['train_len'] == tl) & (df['test_len'] == test_l)]
        if len(rows) == 0:
            print(f'  {"---":>6s}', end='')
        else:
            adv = rows['adv'].mean()
            print(f'  {adv:>+.3f}', end='')
    print()

## Como interpretar

O heatmap mostra a **vantagem do HoTHP** (NLL_RoTHP - NLL_HoTHP) para cada combinacao de TRAIN_LEN e fator de extrapolacao.

- Verde = HoTHP melhor (vantagem positiva)
- Vermelho = RoTHP melhor (vantagem negativa)
- Estrelas = significancia estatistica

O objetivo eh encontrar o **limiar** a partir do qual a vantagem do HoTHP aparece de forma significativa. Esse limiar indica o comprimento minimo de sequencia e fator de extrapolacao necessarios para que o kernel hiperbolico faca diferenca na pratica.